#1 libraries and Tools

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
from io import BytesIO
import statistics
import time
import matplotlib.pyplot as plt

2# Φόρτωση αρχείου από το tarin set

In [ ]:
parquet_file = r'C:\Users\user\Desktop\pythonikos\train-00000-of-00001.parquet'

df = pd.read_parquet(parquet_file, engine='auto')

parquet_file = r'C:\Users\user\Desktop\pythonikos\test-00000-of-00001.parquet'

dftest = pd.read_parquet(parquet_file, engine='auto')

#3 μετατροπή απο png σε byte vector 

In [ ]:
peiramatiko_sunolo = 60000
peiramatiko_sunolo_test = 10000 

peiramatiki_lista = []
labels_lista_train = []

for i in range(peiramatiko_sunolo):
    raw_bytes = df.iloc[i]["image"]["bytes"]

    img = Image.open(BytesIO(raw_bytes))
    img = img.convert("L")

    arr = np.array(img)
    vector = arr.reshape(-1)
    peiramatiki_lista.append(vector)

    labels_lista_train.append(df.iloc[i]["label"])


lista_test = []
labels_lista_test = []
for i in range(peiramatiko_sunolo_test):
    raw_bytes = dftest.iloc[i]["image"]["bytes"]

    img = Image.open(BytesIO(raw_bytes))
    img = img.convert("L")

    arr = np.array(img)
    vector = arr.reshape(-1)
    lista_test.append(vector)

    labels_lista_test.append(dftest.iloc[i]["label"])

X_train = np.array(peiramatiki_lista)

X_test = np.array(lista_test)

X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0




ευκλειδια

In [5]:
def euklidia(test, kentro):
    return np.sqrt(np.sum((test - kentro) ** 2))

Manhattan 

In [6]:
def Manhattan(test, kentro):
    sum = 0
    for i in range(len(test)):
        sum += abs(test[i] - kentro[i])
    return sum

δημιουργούμε κέντρα για την ευκλείδια

In [ ]:
mesoi_oroi_synola = np.zeros((10, 784))
liasta_gia_dieraiti = np.zeros(10)

for i in range(peiramatiko_sunolo):
    liasta_gia_dieraiti[labels_lista_train[i]] +=1                      
    mesoi_oroi_synola[labels_lista_train[i]] += X_train[i]              

centroids = np.zeros((10, 784))                                                

for i in range(10):
    if liasta_gia_dieraiti[i] > 0:
        centroids[i] = mesoi_oroi_synola[i] / liasta_gia_dieraiti[i]

Ευκλείδια απόσταση των test απο τα φτιαγμένα centroids

In [ ]:
start_time = time.perf_counter()

sum_epitixiwn = 0

swsta_paradeigmata = {}
lathws_paradeigmata = {}

pinakas_syn = np.zeros((10,10), dtype=int)


for j in range(peiramatiko_sunolo_test):   # 10.000
    min = float('inf')

    for i in range(10):
        twrini_euklidia = euklidia(X_test[j], centroids[i])
        if twrini_euklidia < min:
            min = twrini_euklidia
            apotel = i

    if apotel == labels_lista_test[j]:
        sum_epitixiwn += 1
    pinakas_syn[labels_lista_test[j], apotel] += 1
        
    if apotel not in swsta_paradeigmata:
        swsta_paradeigmata[apotel] = X_test[j].reshape(28, 28)
    
    if apotel not in lathws_paradeigmata:
        lathws_paradeigmata[labels_lista_test[j]] = (X_test[j].reshape(28, 28), apotel)
    

end_time = time.perf_counter()

print(sum_epitixiwn)
print('Time: ', end_time - start_time)

df_cm = pd.DataFrame(pinakas_syn, index=range(10), columns=range(10))

print("Πίνακας Σύγχυσης:")
print(df_cm)

plt.figure(figsize=(12, 6))
for i in range(10):
    if i in lathws_paradeigmata:
        img, pred = lathws_paradeigmata[i]
        plt.subplot(2, 5, i+1)
        plt.imshow(img, cmap='gray')
        plt.title(f"Real: {i}, Pred: {int(pred)}")
        plt.axis('off')
plt.show()


8203
Time:  0.5670906000304967


knn 

In [12]:
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

sum_epitixiwn = 0

K = 5

paradeigmata = []
paradeigmata_provlepseis = []
paradeigmata_pragmatika = []

pinakas_syn = np.zeros((10,10), dtype=int)

start_time = time.perf_counter()

for i in range(peiramatiko_sunolo_test):
    
    kontinoteroi_apostaseis = np.full(K, np.inf, dtype=np.float32)                                                               
    kontinoteroi_labels = np.zeros(K)

    for j in range(peiramatiko_sunolo):

        d = np.sum((X_train[j] - X_test[i]) ** 2)

        if d < kontinoteroi_apostaseis.max():
            topikomax = np.argmax(kontinoteroi_apostaseis)
            kontinoteroi_labels[topikomax] = labels_lista_train[j]
            kontinoteroi_apostaseis[topikomax] = d 
    
    
    pinakas_syn[labels_lista_test[i], int(statistics.mode(kontinoteroi_labels))] += 1

    if statistics.mode(kontinoteroi_labels) == labels_lista_test[i]:
        sum_epitixiwn +=1
    else:
        if i > 50:
            paradeigmata.append(X_test[i].reshape(28,28))
            paradeigmata_provlepseis.append(statistics.mode(kontinoteroi_labels))
            paradeigmata_pragmatika.append(labels_lista_test[i])
        if len(paradeigmata) > 4:  
            break



print(sum_epitixiwn)

df_cm = pd.DataFrame(pinakas_syn, index=range(10), columns=range(10))

print("Πίνακας Σύγχυσης:")
print(df_cm)
plt.figure(figsize=(12, 4))
for i in range(4):
        img= paradeigmata[i]
        plt.subplot(2, 5, i+1)
        plt.imshow(img, cmap='gray')
        plt.title(f"Real {paradeigmata_pragmatika[i]}, Pred: {int(paradeigmata_provlepseis[i])}")
        plt.axis('off')
plt.show()

KeyboardInterrupt: 

Εύρεση τπυ threashold Βάζεις όλες αυτές τις αποστάσεις (60.000 τιμές) σε μια μεγάλη λίστα και τις ταξινομείς από τη μικρότερη στη μεγαλύτερη.

Κατώφλι (Threshold): Το 70ο ποσοστημόριο είναι η τιμή εκείνη που αφήνει το 70% των αποστάσεων από κάτω της.

Παράδειγμα: Αν το 70% των εικόνων έχουν απόσταση μικρότερη από 2100, τότε το 2100 είναι το Threshold σου.

In [9]:
apostaseis_olwn = []

for j in range(len(X_train)):
    apostaseis_olwn.append(euklidia(X_train[j], centroids[labels_lista_train[j]]))

threashold = np.percentile(apostaseis_olwn, 70)
print(threashold)

6.92495048979021


Η ιδέα μου για ένα Hybrid αλγόριθμο που συνδιάζει τον αλγόριθμο των centroids και K-NN με χρήση γκρίζας ζώνης 

In [ ]:
sum_epitixiwn = 0
start_time = time.perf_counter()
pinakas_syn = np.zeros((10,10), dtype=int)

for j in range(peiramatiko_sunolo_test):   # 10.000
    min = float('inf')
    apotel = -1

    for i in range(10):
        twrini_euklidia = np.sum((X_test[j] - centroids[i]) ** 2)
        if twrini_euklidia < min:
            min = twrini_euklidia
            apotel = i

    if min > 6.9:                                #edw einai h aktina gia thn griza zvnh 
        k = 3
    
        kontinoteroi_apostaseis = np.full(k, np.inf, dtype=np.float32)                                                               
        kontinoteroi_labels = np.zeros(k)

        max_dist_in_k = np.inf

        for siggrisi in range(peiramatiko_sunolo):

            d = np.sum((X_train[siggrisi] - X_test[j]) ** 2)

            if d < max_dist_in_k:
                topikomax = np.argmax(kontinoteroi_apostaseis)
                kontinoteroi_labels[topikomax] = labels_lista_train[siggrisi]
                kontinoteroi_apostaseis[topikomax] = d 
                max_dist_in_k = kontinoteroi_apostaseis.max()

        apotel = statistics.mode(kontinoteroi_labels)

    if apotel == labels_lista_test[j]:
        sum_epitixiwn += 1
    pinakas_syn[labels_lista_test[j]][int(apotel)] += 1

    if j % 1000 == 0:
        print(j)
        print(time.ctime())

print(sum_epitixiwn)
end_time = time.perf_counter()
print('Time: ', end_time - start_time)

df_cm = pd.DataFrame(pinakas_syn, index=range(10), columns=range(10))

print("Πίνακας Σύγχυσης:")
print(df_cm)                                   

0
Fri Mar 27 13:29:14 2026
1000
Fri Mar 27 13:33:29 2026
2000
Fri Mar 27 13:38:01 2026
3000
Fri Mar 27 13:42:08 2026
4000
Fri Mar 27 13:46:14 2026
5000
Fri Mar 27 13:50:19 2026
6000
Fri Mar 27 13:54:24 2026
7000
Fri Mar 27 13:58:29 2026
8000
Fri Mar 27 14:03:06 2026
9000
Fri Mar 27 14:07:31 2026
9693
Time:  2562.62047059997
Πίνακας Σύγχυσης:
     0     1    2    3    4    5    6    7    8    9
0  973     1    1    0    0    1    3    1    0    0
1    0  1130    2    0    0    0    2    0    0    1
2   11     7  988    2    1    0    2   17    4    0
3    0     3    2  973    1   14    1    8    4    4
4    1     7    0    0  946    0    5    1    1   21
5    5     0    0    6    2  865    6    2    2    4
6    6     3    0    0    3    2  944    0    0    0
7    0    24    3    0    1    0    0  990    0   10
8    6     3    6   11    7   11    1    6  918    5
9    5     6    3    5    7    3    1   11    2  966
